In [ ]:
from pathlib import Path
from cv2 import imshow
from monai.config import PathLike
from monai.data import CacheDataset, DataLoader, Dataset, DistributedSampler, SmartCacheDataset, load_decathlon_datalist
from monai.transforms import (
    EnsureChannelFirstd,
    Compose,
    CropForegroundd,
    LoadImaged,
    Orientationd,
    RandSpatialCropSamplesd,
    ScaleIntensityRanged,
    SpatialPadd,
    ToTensord,
)
from utils.data_utils import get_custom_datalist
import matplotlib.pyplot as plt

datalist, val_files = get_custom_datalist(
    "/home/czfy/AS_MAE_Data/nii_origin_instance_number_split_20241209/", seq="T1"
)
transform_1 = Compose(
    [
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys=["image"]),
    ]
)
print(datalist[0])
x_1 = transform_1(datalist[0])
print(x_1["image"].shape)
plt.imshow(x_1["image"][0][..., 0], cmap="gray")


In [ ]:
transform_2 = Orientationd(keys=["image"], axcodes="RAS")
x_2 = transform_2(x_1)
print(x_2["image"].shape)
plt.imshow(x_2["image"][0][...,0].flip(0,1), cmap="gray")
print(x_2["image"].meta["original_affine"])
print(x_2["image"].meta["affine"])

In [ ]:
import torch
print(torch.tensor(x_2["image"].meta["original_affine"]).to(float) @ torch.tensor([0,0,0,1.]).to(float))
print(x_2["image"].meta["affine"].to(float)@torch.tensor([872,872,0,1]).to(float))

In [ ]:
# test RandSpatialCropSamplesd
from monai.transforms import RandSpatialCropSamplesd,ToTensord
# x_2 = transform_1(datalist[:5])
sample =Compose([
     RandSpatialCropSamplesd(
    keys=["image"],
    roi_size=[96, 96, 96],
    num_samples=3,
    random_center=True,
),ToTensord(keys=["image"])])
# print(x_2.size())
sampled = sample(x_1)
print(sampled[0]["image"].shape)
# print(len(sampled))

In [ ]:
sampled[0]["image"].shape


In [ ]:
# test state_dict
import os
import sys
import argparse
from models.ssl_head import SSLHead

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# sys.argv = ["train.py", "--seq", "T1"]

parser = argparse.ArgumentParser(description="PyTorch Training")
parser.add_argument("--logdir", default="test", type=str, help="directory to save the tensorboard logs")
parser.add_argument("--epochs", default=100, type=int, help="number of training epochs")
parser.add_argument("--num_steps", default=100000, type=int, help="number of training iterations")
parser.add_argument("--eval_num", default=100, type=int, help="evaluation frequency")
parser.add_argument("--warmup_steps", default=500, type=int, help="warmup steps")
parser.add_argument("--in_channels", default=1, type=int, help="number of input channels")
parser.add_argument("--feature_size", default=48, type=int, help="embedding size")
parser.add_argument("--dropout_path_rate", default=0.0, type=float, help="drop path rate")
parser.add_argument("--use_checkpoint", action="store_true", help="use gradient checkpointing to save memory")
parser.add_argument("--spatial_dims", default=3, type=int, help="spatial dimension of input data")
parser.add_argument("--a_min", default=-1000, type=float, help="a_min in ScaleIntensityRanged")
parser.add_argument("--a_max", default=1000, type=float, help="a_max in ScaleIntensityRanged")
parser.add_argument("--b_min", default=0.0, type=float, help="b_min in ScaleIntensityRanged")
parser.add_argument("--b_max", default=1.0, type=float, help="b_max in ScaleIntensityRanged")
parser.add_argument("--space_x", default=1.5, type=float, help="spacing in x direction")
parser.add_argument("--space_y", default=1.5, type=float, help="spacing in y direction")
parser.add_argument("--space_z", default=2.0, type=float, help="spacing in z direction")
parser.add_argument("--roi_x", default=256, type=int, help="roi size in x direction")
parser.add_argument("--roi_y", default=256, type=int, help="roi size in y direction")
parser.add_argument("--roi_z", default=12, type=int, help="roi size in z direction")
parser.add_argument("--batch_size", default=1, type=int, help="number of batch size")
parser.add_argument("--sw_batch_size", default=2, type=int, help="number of sliding window batch size")
parser.add_argument("--lr", default=4e-4, type=float, help="learning rate")
parser.add_argument("--decay", default=0.1, type=float, help="decay rate")
parser.add_argument("--momentum", default=0.9, type=float, help="momentum")
parser.add_argument("--lrdecay", action="store_true", help="enable learning rate decay")
parser.add_argument("--max_grad_norm", default=1.0, type=float, help="maximum gradient norm")
parser.add_argument("--loss_type", default="SSL", type=str)
parser.add_argument("--opt", default="adamw", type=str, help="optimization algorithm")
parser.add_argument("--lr_schedule", default="warmup_cosine", type=str)
parser.add_argument("--resume", default=None, type=str, help="resume training")
parser.add_argument("--load_from", default=None, type=str, help="load model from a checkpoint")
parser.add_argument("--local_rank", type=int, default=0, help="local rank")
parser.add_argument("--grad_clip", action="store_true", help="gradient clip")
parser.add_argument("--noamp", action="store_true", help="do NOT use amp for training")
parser.add_argument("--dist-url", default="env://", help="url used to set up distributed training")
parser.add_argument("--smartcache_dataset", action="store_true", help="use monai smartcache Dataset")
parser.add_argument("--cache_dataset", action="store_true", help="use monai cache Dataset")
parser.add_argument("--persistent_dataset", action="store_true", help="use monai persistent Dataset")
parser.add_argument("--seq", required=True, type=str, help="choose between T1, T2 and FS")
args = parser.parse_args([
    "--use_checkpoint", "--num_steps=100000", "--lrdecay", "--eval_num=500",
    "--lr=6e-6", "--decay=0.1", "--seq=T1", "--persistent_dataset",
    "--load_from=/home/czfy/AS_MAE/log/model_swinvit.pt", "--logdir=pretrain"
])

import torch
model = SSLHead(args=parser.parse_args())

local_state_dict = model.state_dict().keys()

model_pth = args.load_from
prepared_state_dict = torch.load(model_pth, map_location="cuda")["state_dict"].keys()

In [ ]:
from copy import deepcopy
prepared_state_dict_copy = list(prepared_state_dict).copy()
for k in local_state_dict:
    if k.replace("swinViT.", "module.").replace("linear",'fc') not in prepared_state_dict:
        print(k)
    else:
        prepared_state_dict_copy.remove(k.replace("swinViT.","module.").replace("linear",'fc'))
print(len(prepared_state_dict_copy))
for i in prepared_state_dict_copy:
    print(i)

In [ ]:
torch.load(model_pth, map_location="cuda")["state_dict"]["module.convTrans3d.weight"].shape


In [26]:
model = SSLHead(args=parser.parse_args(),upsample="large_kernel_deconv")
map_keys = {k:("module."+k.replace('swinViT.',"")) for k in model.state_dict().keys()}

In [ ]:
model_dict = torch.load(model_pth, map_location="cuda")
map_keys = {
    ("module." + k.replace("swinViT.", "").replace("linear", "fc")): k for k in model.state_dict().keys()
}
map_keys.update({"module.convTrans3d.weight":"conv.weight"})
map_keys.update({"module.convTrans3d.bias":"conv.bias"})
model_dict["state_dict"] = {map_keys[k]:v for k,v in model_dict["state_dict"].items() if k in map_keys}
model.load_state_dict(model_dict["state_dict"])

In [ ]:
model_dict = torch.load(model_pth, map_location="cuda")
print(set(map_keys.keys())-set(model_dict["state_dict"].keys()))
print(set(model_dict["state_dict"].keys())-set(map_keys.keys()))
